### Fixing the Receptor PDBQT File
The previous error indicated that the receptor PDBQT file contained 'BRANCH' tags, which are typically associated with flexible residues. Vina expects rigid receptors when `set_receptor` is called with a single argument. To fix this, we will use `Open Babel` to convert the `protein_receptor_fixed.pdbqt` file to a truly rigid format by removing any flexible parts, and save it as `receptor_rigid.pdbqt`.

In [ ]:

# Install Open Babel if not already installed
!pip install vina
!apt-get update -qq
!apt-get install -y openbabel

In [ ]:
# Convert the receptor PDBQT to a rigid format using obabel
# This command attempts to remove flexible residues and create a rigid receptor.
# The -xr option extracts a rigid receptor.
# The -opdbqt specifies the output format as PDBQT.
# The -O specifies the output filename.
!obabel /content/protein_receptor_fixed.pdbqt -xr -opdbqt -O /content/receptor_rigid.pdbqt

print('Fixed receptor file created as /content/receptor_rigid.pdbqt')

In [ ]:
from vina import Vina

# 1. Initialize the Vina docking engine
# 'sf_name' specifies the scoring function (Vina is the standard)
v = Vina(sf_name='vina')

# 2. Load your structural files
# Note: For the receptor, use the clean version without ligand fragments
v.set_receptor('/content/receptor_rigid.pdbqt') # Changed to use the newly created rigid receptor
v.set_ligand_from_file('/candidate_51003064.pdbqt')

# 3. Define the Search Space (The BCL-2 Binding Pocket)
# We use the exact center coordinates we found for the 6O0K active site
v.compute_vina_maps(
    center=[-17.4, 7.3, -18.2],
    box_size=[20.0, 20.0, 20.0]
)

# 4. Run the simulation
# exhaustiveness controls how thoroughly it searches the pocket (8 is standard)
print("[*] Simulating molecular docking interactions...")
v.dock(exhaustiveness=8, n_poses=5)

# 5. Extract and print the best score
energies = v.energies()
best_score = energies[0][0]  # The first pose contains the lowest free energy

print("\n" + "="*35)
print(f"  FINAL DOCKING SCORE: {best_score:.2f} kcal/mol")
print("="*35)

# 6. Save the results (where the molecule physically landed)
v.write_poses('docking_outputs.pdbqt', n_poses=5, overwrite=True)

In [ ]:
# Install py3Dmol for 3D visualization
!pip install py3Dmol

import py3Dmol

# Create a 3Dmol viewer object
view = py3Dmol.view(width=800, height=600)

# Load the rigid receptor
with open('/content/receptor_rigid.pdbqt', 'r') as f_receptor:
    receptor_data = f_receptor.read()
view.addModel(receptor_data, 'pdbqt')

# Load the docked ligand poses
with open('/content/docking_outputs.pdbqt', 'r') as f_ligand:
    ligand_data = f_ligand.read()
view.addModel(ligand_data, 'pdbqt')

# Style the receptor
view.setStyle({'model': 0}, {'cartoon': {'color': 'spectrum'}})

# Style the ligand (all poses in the docking_outputs.pdbqt file)
# Using 'stick' style for clarity and coloring by atom type
view.setStyle({'model': 1}, {'stick': {'colorscheme': 'carbon'}})

# Zoom to fit all models and render the view
view.zoomTo()
view.show()

In [ ]:

# CHEMBL1824

!pip install chembl_webresource_client pandas

In [ ]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

# Target: Receptor tyrosine-protein kinase erbB-2
target_id = "CHEMBL1824"
activity = new_client.activity

In [ ]:
# Fetching activities related directly to the target
res = activity.filter(target_chembl_id=target_id).filter(standard_type="IC50")
df = pd.DataFrame.from_dict(res)

print(f"Total rows retrieved: {len(df)}")

In [ ]:
import pandas as pd

# Filter the original 'df' for human targets
# Assuming 'target_organism' column exists in 'df' and contains 'Homo sapiens' for human data
df_human_filtered = df[df['target_organism'] == 'Homo sapiens'].copy()

# Define columns to keep (already defined in previous cell but good to ensure here)
columns_to_keep = [
    'molecule_chembl_id',
    'canonical_smiles',
    'standard_type',
    'standard_value',
    'standard_units',
    'pchembl_value'
]

# Create the cleaned dataframe for human targets
clean_df = df_human_filtered[columns_to_keep].dropna(subset=['canonical_smiles', 'standard_value'])

# Save the human-specific dataset to a new CSV file
output_filename = 'CHEMBL1824_HER2_human_dataset.csv'
clean_df.to_csv(output_filename, index=False)
print(f"Dataset for human targets saved successfully to '{output_filename}'!")
print(f"Total rows for human targets retrieved: {len(clean_df)}")

# Display the first few rows of the filtered dataframe
clean_df.head()

In [ ]:
human_df = pd.read_csv('/content/CHEMBL1824_HER2_human_dataset.csv')
human_df

### 1. Data Preprocessing: Handling and Standardizing `pchembl_value`

Sometimes `pchembl_value` might be missing or need recalculation based on `standard_value`. The ChEMBL database usually provides `pchembl_value` when the `standard_type` is an IC50, Ki, or Kd, and the `standard_value` is in nM. We can ensure all relevant entries have a `pchembl_value`.

First, let's verify the data types and distribution of `pchembl_value`.

In [ ]:
# Check data types and basic statistics for pchembl_value
print(human_df['pchembl_value'].info())
print(human_df['pchembl_value'].describe())

# Handle potential missing pchembl_value by converting standard_value (nM to pIC50)
# pIC50 = -log10(IC50 * 1e-9) where IC50 is in nM
def calculate_pic50(row):
    if pd.isna(row['pchembl_value']) and row['standard_type'] == 'IC50' and row['standard_units'] == 'nM':
        # Convert nM to M, then take -log10
        return -np.log10(row['standard_value'] * 1e-9)
    else:
        return row['pchembl_value']

import numpy as np
human_df['pchembl_value'] = human_df.apply(calculate_pic50, axis=1)

# Drop rows where pchembl_value is still NaN (if any other standard_type or units were present and not handled)
human_df.dropna(subset=['pchembl_value'], inplace=True)

print(f"Total rows after pchembl_value standardization: {len(human_df)}")
display(human_df.head())

### 2. Molecular Descriptors Calculation using RDKit

Next, we'll calculate molecular descriptors from the `canonical_smiles` column. We will use the `rdkit` library, a powerful cheminformatics toolkit.

First, we need to install `rdkit`.

In [ ]:
!pip install rdkit

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np

# Function to calculate molecular descriptors from SMILES
def get_molecular_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Example descriptors: LogP, Molecular Weight, Number of H-bond donors/acceptors
    descriptors = {
        'MolLogP': Descriptors.MolLogP(mol),
        'MolWt': Descriptors.MolWt(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'TPSA': Descriptors.TPSA(mol)
    }
    return descriptors

# Apply the function to the 'canonical_smiles' column
# This might take a while for large datasets
df_descriptors = human_df['canonical_smiles'].apply(get_molecular_descriptors)

# Convert the series of dictionaries into a DataFrame
df_descriptors = pd.DataFrame(df_descriptors.tolist())

# Combine the descriptors with the original dataframe
human_df = pd.concat([human_df, df_descriptors], axis=1)

# Drop rows where descriptor calculation might have failed (if any smiles were invalid)
human_df.dropna(subset=df_descriptors.columns, inplace=True)

print(f"Total rows after descriptor calculation: {len(human_df)}")
display(human_df.head())

### 3. Molecular Descriptors Calculation using PaDEL

Now, let's apply the `smiles_to_padel_features` function to calculate additional molecular descriptors using `padelpy`.

In [ ]:
# Install padelpy if not already installed
!pip install padelpy

In [ ]:
import numpy as np
import pandas as pd
from padelpy import from_smiles

def smiles_to_padel_features(smiles):
    """
    Calculates PaDEL descriptors for a given SMILES string.
    By default, this computes standard 2D descriptors.
    """
    try:
        # Increased timeout to 60 seconds to handle complex molecules
        descriptors = from_smiles(smiles, fingerprints=False, timeout=60)
        return descriptors
    except Exception as e:
        # Catches invalid SMILES or PaDEL processing failures
        # print(f"Error processing SMILES {smiles}: {e}") # Commenting out to reduce log spam
        return None

# Prepare lists to store descriptors and corresponding molecule IDs
padel_descriptors_list = []
molecule_chembl_ids_with_padel = []

print("Calculating PaDEL descriptors... This might take some time.")

# Iterate through each SMILES string in human_df
for idx, row in human_df.iterrows():
    smiles = row['canonical_smiles']
    chembl_id = row['molecule_chembl_id']

    descriptors = smiles_to_padel_features(smiles)

    if descriptors is not None:
        # Add the molecule_chembl_id to the descriptor dictionary for merging
        descriptors['molecule_chembl_id'] = chembl_id
        padel_descriptors_list.append(descriptors)
        molecule_chembl_ids_with_padel.append(chembl_id)

# Create a DataFrame from the collected PaDEL descriptors
# Ensure all descriptor dictionaries have the same keys to avoid NaN columns
# Fill missing keys with None for consistency if from_smiles returns varying sets
if padel_descriptors_list:
    all_keys = sorted(list(set(key for d in padel_descriptors_list for key in d.keys())))
    normalized_descriptors = []
    for d in padel_descriptors_list:
        normalized_descriptors.append({key: d.get(key) for key in all_keys})
    df_padel_descriptors = pd.DataFrame(normalized_descriptors)
    # Convert appropriate columns to numeric, coercing errors to NaN
    for col in df_padel_descriptors.columns:
        if col != 'molecule_chembl_id':
            df_padel_descriptors[col] = pd.to_numeric(df_padel_descriptors[col], errors='coerce')
else:
    df_padel_descriptors = pd.DataFrame()

# Merge the PaDEL descriptors with the original human_df
# Use 'inner' merge to keep only rows for which descriptors were successfully calculated
# Handle case where df_padel_descriptors might be empty or missing columns
if not df_padel_descriptors.empty:
    human_df = pd.merge(
        human_df,
        df_padel_descriptors,
        on='molecule_chembl_id',
        how='inner'
    )
    # Drop rows with any NaN values that might have resulted from 'coerce' during conversion
    human_df.dropna(subset=df_padel_descriptors.columns.drop('molecule_chembl_id', errors='ignore'), inplace=True)

print(f"Total rows after PaDEL descriptor calculation and merge: {len(human_df)}")
display(human_df.head())

In [ ]:
# Select critical columns
columns_to_keep = [
    'molecule_chembl_id',
    'canonical_smiles',
    'standard_type',
    'standard_value',
    'standard_units',
    'pchembl_value'
]

# Filter and drop empty structures/values
clean_df = df[columns_to_keep].dropna(subset=['canonical_smiles', 'standard_value'])

# Save to file
clean_df.to_csv('CHEMBL1824_HER2_dataset.csv', index=False)
print("Dataset saved successfully to 'CHEMBL1824_HER2_dataset.csv'!")

In [ ]:
clean_df